# Lectura de Códigos QR con OpenCV y PyZbar

QR = Quick Response

## Introducción

En esta práctica aprenderemos a implementar un sistema de lectura de códigos QR en tiempo real utilizando la webcam. Combinaremos dos bibliotecas fundamentales:

- **OpenCV (cv2)**: Para captura de video y manipulación de imágenes
- **PyZbar**: Para decodificación específica de códigos QR y códigos de barras

## Objetivos de Aprendizaje

Al finalizar esta práctica serás capaz de:
- Capturar video desde la webcam usando OpenCV
- Detectar y decodificar códigos QR en tiempo real
- Superponer información visual sobre los códigos detectados
- Implementar un bucle de video con control de salida

## Conceptos Teóricos

### ¿Qué es un Código QR?
Los códigos QR (Quick Response) son códigos de barras bidimensionales que pueden almacenar diversos tipos de información:
- URLs
- Texto plano
- Números de teléfono
- Información de contacto
- Coordenadas geográficas

### Cómo puedo generarlo
En otro apartado haremos un generado rde código QR por código, pero para testear y probar esto con QRs propios puedes buscar por internet "QR Generator" o dierectametente en Google Chrome (o cualquier otro navegador basado en Chromium) hacer botón derecho y ver el QR de la página en la que te encuentres.

### Cómo funciona un código QR
Video corto: https://www.youtube.com/watch?v=w0hWyJw3Y8Y
Video largo: https://www.youtube.com/watch?v=4XTkiudd-_E

### Bibliotecas Necesarias

```python
import cv2          # OpenCV para procesamiento de video e imágenes
from pyzbar import pyzbar  # PyZbar para decodificación de códigos QR/barras
```

**Instalación:**
```bash
pip install opencv-python
pip install pyzbar
```

## Código Completo Explicado

```python
import cv2
from pyzbar import pyzbar

# Inicializar la webcam
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Detectar códigos QR en la imagen
    qr_codes = pyzbar.decode(frame)

    for qr in qr_codes:
        # Extraer bounding box (polígono)
        x, y, w, h = qr.rect
        cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 3)

        # Decodificar contenido
        qr_data = qr.data.decode("utf-8")
        qr_type = qr.type

        # Mostrar texto sobre la imagen
        text = f"{qr_type}: {qr_data}"
        cv2.putText(frame, text, (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX,
                    0.7, (255, 0, 0), 2)

    # Mostrar resultado en ventana
    cv2.imshow("Lector QR - Webcam", frame)

    # Salir con tecla 'q'
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
```

## Análisis Línea por Línea

### 1. Inicialización de la Webcam
```python
cap = cv2.VideoCapture(0)
```
- Crea un objeto VideoCapture para acceder a la cámara
- El parámetro `0` indica la primera cámara disponible
- Para cámaras externas usar `1`, `2`, etc.

### 2. Bucle Principal de Captura
```python
while True:
    ret, frame = cap.read()
    if not ret:
        break
```
- `ret`: Variable booleana que indica si la captura fue exitosa
- `frame`: Matriz NumPy que contiene la imagen capturada
- Si no hay frame disponible, se sale del bucle

### 3. Detección de Códigos QR
```python
qr_codes = pyzbar.decode(frame)
```
- `pyzbar.decode()` analiza la imagen completa buscando códigos
- Retorna una lista con todos los códigos detectados
- Cada elemento contiene información sobre posición, tipo y contenido

### 4. Procesamiento de Códigos Detectados
```python
for qr in qr_codes:
    x, y, w, h = qr.rect
    cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 3)
```
- `qr.rect` proporciona las coordenadas del rectángulo contenedor
- `cv2.rectangle()` dibuja un marco verde alrededor del código detectado
- Color en formato BGR: `(0, 255, 0)` = verde

### 5. Extracción del Contenido
```python
qr_data = qr.data.decode("utf-8")
qr_type = qr.type
```
- `qr.data` contiene los datos en formato bytes
- `.decode("utf-8")` convierte a string legible
- `qr.type` indica el tipo de código (QRCODE, CODE128, etc.)

### 6. Visualización de la Información
```python
text = f"{qr_type}: {qr_data}"
cv2.putText(frame, text, (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX,
            0.7, (255, 0, 0), 2)
```
- Crea un string con el tipo y contenido del código
- `cv2.putText()` superpone texto en la imagen
- Posición: `(x, y - 10)` = justo arriba del rectángulo
- Color azul: `(255, 0, 0)` en formato BGR

### 7. Control de Salida
```python
if cv2.waitKey(1) & 0xFF == ord('q'):
    break
```
- `cv2.waitKey(1)` espera 1ms por una tecla presionada
- `& 0xFF` extrae solo los 8 bits menos significativos
- Compara con el código ASCII de 'q' para salir

## Ejercicios Propuestos

### Ejercicio 1: Contador de QR
Modifica el código para contar cuántos códigos QR diferentes has escaneado en la sesión.

### Ejercicio 2: Historial de Códigos
Implementa un historial que muestre los últimos 5 códigos QR detectados en pantalla.

### Ejercicio 3: Filtro por Tipo
Añade funcionalidad para detectar solo códigos QR (ignorar códigos de barras).

### Ejercicio 4: Guardado de Datos
Guarda automáticamente el contenido de los códigos QR en un archivo de texto con timestamp.

## Posibles Mejoras

1. **Detección más robusta**: Convertir a escala de grises para mejorar detección
2. **Feedback visual**: Cambiar color del rectángulo según el tipo de código
3. **Validación de URLs**: Verificar si el contenido es una URL válida
4. **Beep sonoro**: Reproducir sonido cuando se detecta un código
5. **Interfaz mejorada**: Añadir botones y controles en la ventana

## Troubleshooting Común

- **Error de cámara**: Verificar que no esté siendo usada por otra aplicación
- **Dependencias**: Instalar `zbar` en sistemas Linux: `sudo apt-get install libzbar0`
- **Rendimiento**: Para cámaras lentas, aumentar el valor en `waitKey()`
- **Detección fallida**: Asegurar buena iluminación y enfoque del código QR

## Conclusiones

Este proyecto demuestra conceptos fundamentales de Computer Vision:
- Captura de video en tiempo real
- Detección de patrones específicos
- Procesamiento de regiones de interés
- Superposición de elementos gráficos
- Interacción usuario-sistema

In [ ]:
# Instalar librerías
%pip install opencv-python pyzbar


In [ ]:
import cv2
from pyzbar import pyzbar

# Inicializar la webcam
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Detectar códigos QR en la imagen
    qr_codes = pyzbar.decode(frame)

    for qr in qr_codes:
        # Extraer bounding box (polígono)
        x, y, w, h = qr.rect
        cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 3)

        # Decodificar contenido
        qr_data = qr.data.decode("utf-8")
        qr_type = qr.type

        # Mostrar texto sobre la imagen
        text = f"{qr_type}: {qr_data}"
        cv2.putText(frame, text, (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX,
                    0.7, (255, 0, 0), 2)

    # Mostrar resultado en ventana
    cv2.imshow("Lector QR - Webcam", frame)

    # Salir con tecla 'q'
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
